# Layer 5 — Web Application Deployment

Deploys the Guwahati Urban Prediction web app:
1. **Generate tile manifest** — refresh GEE tile URLs from Layer 4 pipeline
2. **Bundle app** — copy index.html + manifest + PMTiles to deploy directory
3. **Local preview** — serve via Python HTTP server in Colab
4. **Deploy** — push to Firebase Hosting or download as ZIP

The app is a single-page MapLibre GL site. No build step needed.

## 0. Setup & run Layer 4 pipeline

In [ ]:
# This cell runs the FULL Layer 4 pipeline to generate fresh tile URLs
# It's self-contained — no need to run notebooks 01-04 first

import ee, json, os, shutil
import numpy as np

ee.Authenticate()
ee.Initialize(project='vivid-brand-182819')
print('Earth Engine initialized')

BBOX = {'west': 91.55, 'south': 26.10, 'east': 91.88, 'north': 26.25}
aoi = ee.Geometry.Rectangle([BBOX['west'], BBOX['south'], BBOX['east'], BBOX['north']])
SCALE = 30
LULC_PALETTE = ['e41a1c', '4daf4a', '377eb8', 'c2a06a']
LULC_LABELS = ['Urban', 'Vegetation', 'Water', 'Barren']
CENTER = [26.1445, 91.7362]

# ── Composite builders ──
def cloud_mask_landsat(image):
    qa = image.select('QA_PIXEL')
    return image.updateMask(qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0)))

def add_indices(image, nir, red, swir, green):
    return image.addBands([
        image.normalizedDifference([nir, red]).rename('NDVI'),
        image.normalizedDifference([swir, nir]).rename('NDBI'),
        image.normalizedDifference([green, nir]).rename('NDWI'),
    ])

def scale_l8(image):
    return image.addBands(image.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']).multiply(0.0000275).add(-0.2), overwrite=True)

def scale_l5(image):
    return image.addBands(image.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7']).multiply(0.0000275).add(-0.2), overwrite=True)

def build_composite(start, end, sensor):
    if sensor == 'L8':
        col_id, sfn, bands = 'LANDSAT/LC08/C02/T1_L2', scale_l8, ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']
        nir, red, swir, green = 'SR_B5', 'SR_B4', 'SR_B6', 'SR_B3'
    else:
        col_id, sfn, bands = 'LANDSAT/LT05/C02/T1_L2', scale_l5, ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7']
        nir, red, swir, green = 'SR_B4', 'SR_B3', 'SR_B5', 'SR_B2'
    _n, _r, _s, _g = nir, red, swir, green
    def _add(img, _n=_n, _r=_r, _s=_s, _g=_g): return add_indices(img, _n, _r, _s, _g)
    col = (ee.ImageCollection(col_id).filterBounds(aoi).filterDate(start, end)
           .filter(ee.Filter.lt('CLOUD_COVER', 30)).map(cloud_mask_landsat).map(sfn).map(_add))
    return col.select(bands + ['NDVI','NDBI','NDWI']).median().clip(aoi).toFloat()

def proximity_stack(lulc_img):
    dw = lulc_img.eq(2).fastDistanceTransform(256).sqrt().multiply(SCALE).rename('dist_water')
    edge = lulc_img.eq(0).And(lulc_img.eq(0).focal_min(1).Not())
    de = edge.fastDistanceTransform(256).sqrt().multiply(SCALE).rename('dist_urban_edge')
    viirs = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG').filterBounds(aoi).filterDate('2021-01-01','2021-12-31').select('avg_rad').median().clip(aoi)
    lit = viirs.gt(5).unmask(0)
    dr = lit.fastDistanceTransform(256).sqrt().multiply(SCALE).rename('dist_road')
    srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
    return dw.addBands(de).addBands(dr).addBands(ee.Terrain.slope(srtm).rename('slope')).addBands(srtm.select('elevation')).toFloat()

# ── Build composites ──
EPOCHS_CFG = {
    '2000':('2000-11-01','2001-04-30','L5'), '2005':('2005-11-01','2006-04-30','L5'),
    '2010':('2010-11-01','2011-04-30','L5'), '2016':('2016-11-01','2017-04-30','L8'),
    '2021':('2021-11-01','2022-04-30','L8'),
}
composites = {}
for lbl,(s,e,sen) in EPOCHS_CFG.items():
    composites[lbl] = build_composite(s, e, sen)
    print(f'  Composite {lbl}')

# ── Classify ──
CLASSIFY_BANDS = ['NDVI','NDBI','NDWI']
ref = composites['2021']
ndvi, ndbi, ndwi = ref.select('NDVI'), ref.select('NDBI'), ref.select('NDWI')
labels = (ee.Image(0).where(ndbi.gt(0).And(ndvi.lt(0.25)),0).where(ndvi.gt(0.35).And(ndbi.lt(0)),1)
    .where(ndwi.gt(0.1).And(ndvi.lt(0.15)),2).where(ndvi.lt(0.15).And(ndbi.lt(0.05)).And(ndwi.lt(0)),3)
    .rename('label').toUint8())
mask = ndbi.gt(0).And(ndvi.lt(0.25)).Or(ndvi.gt(0.35).And(ndbi.lt(0))).Or(
    ndwi.gt(0.1).And(ndvi.lt(0.15))).Or(ndvi.lt(0.15).And(ndbi.lt(0.05)).And(ndwi.lt(0)))
samples = ref.addBands(labels.updateMask(mask)).stratifiedSample(500,'label',aoi,SCALE,seed=42,geometries=True)
cart = ee.Classifier.smileCart(maxNodes=50).train(samples,'label',CLASSIFY_BANDS)
lulc_maps = {lbl: comp.select(CLASSIFY_BANDS).classify(cart).toUint8().rename('lulc') for lbl,comp in composites.items()}
print('  LULC classified')

# ── Markov + GBT + Forecast ──
def compute_tm(lf, lt):
    tr = lf.multiply(10).add(lt).rename('t'); pa = ee.Image.pixelArea()
    m = np.zeros((4,4))
    for i in range(4):
        for j in range(4):
            m[i][j] = ee.Number(tr.eq(i*10+j).multiply(pa).reduceRegion(ee.Reducer.sum(),aoi,SCALE,maxPixels=1e9).get('t')).getInfo()
    rs = m.sum(axis=1,keepdims=True); rs[rs==0]=1; return m/rs

print('  Computing Markov matrices...')
tms = {}
for e1,e2 in [('2000','2005'),('2005','2010'),('2010','2016'),('2016','2021')]:
    tms[f'{e1}_{e2}'] = compute_tm(lulc_maps[e1], lulc_maps[e2])
avg_tm = np.mean(list(tms.values()), axis=0)

FEAT_BANDS = ['NDVI','NDBI','NDWI','dist_water','dist_urban_edge','dist_road','slope','elevation']
def chg_samples(lf,la,comp,li):
    prox = proximity_stack(li); feat = comp.select(['NDVI','NDBI','NDWI']).addBands(prox)
    bu = lf.neq(0).And(la.eq(0)).rename('became_urban').toUint8()
    sn = lf.neq(0).And(la.neq(0)).rename('became_urban').multiply(0).toUint8()
    return feat.addBands(bu.updateMask(bu)).sample(aoi,SCALE,numPixels=1000,seed=42,geometries=True).merge(
        feat.addBands(sn.updateMask(lf.neq(0).And(la.neq(0)))).sample(aoi,SCALE,numPixels=2000,seed=43,geometries=True))

print('  Training GBT...')
s1 = chg_samples(lulc_maps['2010'],lulc_maps['2016'],composites['2010'],lulc_maps['2010'])
s2 = chg_samples(lulc_maps['2016'],lulc_maps['2021'],composites['2016'],lulc_maps['2016'])
gbt = ee.Classifier.smileGradientTreeBoost(100,shrinkage=0.1,maxNodes=20,seed=42).train(s1.merge(s2),'became_urban',FEAT_BANDS)
gbt_prob = gbt.setOutputMode('PROBABILITY')

def forecast_step(cur, comp, tm, gbt_p):
    prox = proximity_stack(cur); feat = comp.select(['NDVI','NDBI','NDWI']).addBands(prox)
    suit = feat.classify(gbt_p).rename('urban_prob').updateMask(cur.neq(0))
    pa = ee.Image.pixelArea()
    ca = {c: ee.Number(cur.eq(c).multiply(pa).reduceRegion(ee.Reducer.sum(),aoi,SCALE,maxPixels=1e9).get('lulc')).getInfo() for c in range(4)}
    nu = sum(ca[c]*tm[c][0] for c in range(1,4)); nt = sum(ca[c] for c in range(1,4))
    frac = nu/nt if nt>0 else 0; pctl = max(0,min(99,(1-frac)*100))
    thr = ee.Number(suit.reduceRegion(ee.Reducer.percentile([pctl]),aoi,SCALE,maxPixels=1e9).get('urban_prob')).getInfo()
    return cur.where(cur.neq(0).And(suit.gt(thr).unmask(0).And(cur.neq(2))), 0)

predicted = {}; cur = lulc_maps['2021']
for yr in ['2026','2031','2036']:
    print(f'  Forecasting {yr}...')
    cur = forecast_step(cur, composites['2021'], avg_tm, gbt_prob)
    predicted[yr] = cur

# Suitability surface
prox_2021 = proximity_stack(lulc_maps['2021'])
suitability = composites['2021'].select(['NDVI','NDBI','NDWI']).addBands(prox_2021).classify(gbt_prob).rename('urban_prob').toFloat()

print('\nAll layers computed. Generating tile URLs...')

## 1. Write the web application HTML

In [ ]:
# Generate tile URLs and build manifest

tile_urls = {}
lulc_vis = {'min':0, 'max':3, 'palette': LULC_PALETTE}

# LULC — all epochs
all_lulc = {**lulc_maps, **predicted}
for year in sorted(all_lulc.keys(), key=int):
    mi = all_lulc[year].getMapId(lulc_vis)
    tile_urls[f'lulc_{year}'] = mi['tile_fetcher'].url_format

# Urban gain
for e1, e2 in [('2000','2005'),('2005','2010'),('2010','2016'),('2016','2021'),
               ('2021','2026'),('2026','2031'),('2031','2036')]:
    prev = all_lulc.get(e1)
    curr = all_lulc.get(e2)
    if prev and curr:
        gain = prev.neq(0).And(curr.eq(0)).selfMask()
        color = 'ff0000' if int(e2) <= 2021 else 'ff6600'
        mi = gain.getMapId({'palette': [color]})
        tile_urls[f'urban_gain_{e1}_{e2}'] = mi['tile_fetcher'].url_format

# VIIRS
viirs_vis = {'min':0,'max':60,'palette':['000000','0000ff','00ffff','ffff00','ff0000']}
for y in [2016,2019,2021,2023]:
    v = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG').filterBounds(aoi).filterDate(f'{y}-01-01',f'{y}-12-31').select('avg_rad').median().clip(aoi)
    tile_urls[f'viirs_{y}'] = v.getMapId(viirs_vis)['tile_fetcher'].url_format

# Suitability, NDVI, elevation, slope
tile_urls['suitability_2021'] = suitability.getMapId({'min':0,'max':1,'palette':['1a9850','91cf60','d9ef8b','fee08b','fc8d59','d73027']})['tile_fetcher'].url_format
tile_urls['ndvi_2021'] = composites['2021'].select('NDVI').getMapId({'min':-0.1,'max':0.8,'palette':['d73027','fc8d59','fee08b','d9ef8b','91cf60','1a9850']})['tile_fetcher'].url_format
dem = ee.Image('USGS/SRTMGL1_003').clip(aoi)
tile_urls['elevation'] = dem.getMapId({'min':40,'max':300,'palette':['006633','E5FFCC','662A00','D8D8D8','F5F5F5']})['tile_fetcher'].url_format
tile_urls['slope'] = ee.Terrain.slope(dem).getMapId({'min':0,'max':30,'palette':['00ff00','ffff00','ff0000']})['tile_fetcher'].url_format

# Urban area stats
urban_stats = {}
for year in sorted(all_lulc.keys(), key=int):
    a = ee.Number(all_lulc[year].eq(0).multiply(ee.Image.pixelArea()).reduceRegion(
        ee.Reducer.sum(),aoi,SCALE,maxPixels=1e9).get('lulc')).divide(1e6).getInfo()
    urban_stats[year] = round(a, 2)
    print(f'  {year}: {a:.2f} km²')

# Build manifest
manifest = {
    'project': 'Guwahati Urban Development Prediction Platform',
    'version': '1.0.0-poc',
    'center': CENTER, 'zoom': 12,
    'bounds': [BBOX['west'], BBOX['south'], BBOX['east'], BBOX['north']],
    'epochs': {'historical': ['2000','2005','2010','2016','2021'], 'predicted': ['2026','2031','2036']},
    'urban_area_km2': urban_stats,
    'lulc': {'classes': LULC_LABELS, 'palette': ['#'+c for c in LULC_PALETTE]},
    'tile_layers': {
        **{f'lulc_{y}': {'url': tile_urls[f'lulc_{y}'], 'type':'raster', 'group':'lulc',
            'label': f'LULC {y}' + (' (predicted)' if int(y)>2021 else ''), 'opacity':0.7}
           for y in sorted(all_lulc.keys(), key=int)},
        **{k: {'url':v, 'type':'raster', 'group':'change',
            'label': k.replace('urban_gain_','Urban Growth ').replace('_','→'), 'opacity':0.8}
           for k,v in tile_urls.items() if k.startswith('urban_gain')},
        **{f'viirs_{y}': {'url': tile_urls[f'viirs_{y}'], 'type':'raster', 'group':'nightlights',
            'label': f'Night Lights {y}', 'opacity':0.7} for y in [2016,2019,2021,2023]},
        'suitability_2021': {'url': tile_urls['suitability_2021'], 'type':'raster', 'group':'analysis', 'label':'Urban Growth Suitability', 'opacity':0.7},
        'ndvi_2021': {'url': tile_urls['ndvi_2021'], 'type':'raster', 'group':'analysis', 'label':'Vegetation (NDVI)', 'opacity':0.7},
        'elevation': {'url': tile_urls['elevation'], 'type':'raster', 'group':'terrain', 'label':'Elevation (SRTM)', 'opacity':0.6},
        'slope': {'url': tile_urls['slope'], 'type':'raster', 'group':'terrain', 'label':'Slope', 'opacity':0.6},
    },
    'vector_layers': {'osm': {'type':'pmtiles', 'url':'data/osm_guwahati.pmtiles', 'layers':['roads','railways','infrastructure','water']}},
}

print(f'\nManifest: {len(manifest["tile_layers"])} raster layers, {len(tile_urls)} total tile URLs')

## 2. Bundle deployment package

In [ ]:
# Bundle everything into public/ directory for deployment

os.makedirs('public/data', exist_ok=True)

# Save manifest as companion file (backup)
with open('public/tile_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print('Saved: public/tile_manifest.json')

# Write the app HTML with INLINE manifest embedded
# This makes the HTML self-contained — no separate fetch needed
APP_HTML = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Guwahati Urban Prediction</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/maplibre-gl@4.7.1/dist/maplibre-gl.min.css">
<style>
:root{--bg:#1a1a2e;--panel:#16213e;--border:#0f3460;--text:#e0e0e0;--dim:#8a8a9a;--accent:#e94560;--adim:#533483;--urban:#e41a1c;--veg:#4daf4a;--water:#377eb8;--barren:#c2a06a}
*{margin:0;padding:0;box-sizing:border-box}
body{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;background:var(--bg);color:var(--text);overflow:hidden;height:100vh}
#map{position:absolute;inset:0}
#header{position:absolute;top:0;left:0;right:0;z-index:10;background:linear-gradient(180deg,rgba(22,33,62,.95),rgba(22,33,62,.8));backdrop-filter:blur(8px);padding:10px 20px;display:flex;align-items:center;gap:16px;border-bottom:1px solid var(--border)}
#header h1{font-size:16px;font-weight:600;letter-spacing:.5px}
#header .sub{font-size:11px;color:var(--dim)}
#header .badge{font-size:10px;padding:2px 8px;border-radius:10px;background:var(--adim);color:var(--accent);font-weight:600}
#sidebar{position:absolute;top:52px;left:0;bottom:80px;width:280px;z-index:10;background:rgba(22,33,62,.92);backdrop-filter:blur(8px);border-right:1px solid var(--border);overflow-y:auto;padding:12px}
.ps{margin-bottom:16px}
.pt{font-size:10px;font-weight:700;text-transform:uppercase;letter-spacing:1.2px;color:var(--dim);margin-bottom:8px}
.lr{display:flex;align-items:center;gap:8px;padding:4px 8px;font-size:12px}
.ls{width:14px;height:14px;border-radius:3px;flex-shrink:0}
.li{display:flex;align-items:center;gap:8px;padding:6px 8px;border-radius:6px;cursor:pointer;font-size:13px;transition:background .15s}
.li:hover{background:rgba(255,255,255,.05)}
.li input{accent-color:var(--accent)}
#timeline{position:absolute;bottom:0;left:0;right:0;z-index:10;background:rgba(22,33,62,.95);backdrop-filter:blur(8px);border-top:1px solid var(--border);padding:12px 20px 16px}
#tl-label{font-size:24px;font-weight:700;text-align:center;margin-bottom:4px}
#tl-label .st{font-size:11px;font-weight:400;color:var(--dim);margin-left:8px}
#tl-slider{width:100%;-webkit-appearance:none;appearance:none;height:6px;border-radius:3px;background:var(--border);outline:none}
#tl-slider::-webkit-slider-thumb{-webkit-appearance:none;width:18px;height:18px;border-radius:50%;background:var(--accent);cursor:pointer;box-shadow:0 0 8px rgba(233,69,96,.5)}
#tl-ticks{display:flex;justify-content:space-between;padding:4px 0;font-size:10px;color:var(--dim)}
#tl-ticks span{cursor:pointer}
#tl-ticks span:hover{color:var(--text)}
#tl-ticks span.pred{color:var(--adim);font-style:italic}
#stats{position:absolute;top:52px;right:0;width:260px;z-index:10;background:rgba(22,33,62,.92);backdrop-filter:blur(8px);border-left:1px solid var(--border);padding:12px;overflow-y:auto;max-height:calc(100vh - 132px)}
.sc{background:rgba(255,255,255,.03);border:1px solid var(--border);border-radius:8px;padding:12px;margin-bottom:10px}
.sl{font-size:10px;text-transform:uppercase;letter-spacing:1px;color:var(--dim)}
.sv{font-size:22px;font-weight:700;margin-top:2px}
.sd{font-size:11px;margin-top:2px}
.sd.up{color:var(--accent)} .sd.dn{color:var(--veg)}
#chart{width:100%;height:140px;margin-top:8px}
#loading{position:fixed;inset:0;z-index:100;background:var(--bg);display:flex;flex-direction:column;align-items:center;justify-content:center;gap:16px;transition:opacity .5s}
#loading.h{opacity:0;pointer-events:none}
.spin{width:40px;height:40px;border:3px solid var(--border);border-top-color:var(--accent);border-radius:50%;animation:sp .8s linear infinite}
@keyframes sp{to{transform:rotate(360deg)}}
#ls{font-size:12px;color:var(--dim);max-width:80%;text-align:center}
#banner{display:none;position:fixed;top:56px;left:50%;transform:translateX(-50%);z-index:200;background:#ff9800;color:#000;padding:8px 20px;border-radius:0 0 8px 8px;font-size:12px;text-align:center;max-width:90vw;font-weight:500}
@media(max-width:768px){#sidebar{width:220px}#stats{width:200px}}
@media(max-width:520px){#sidebar,#stats{display:none}}
</style>
</head>
<body>
<div id="banner"></div>
<div id="loading"><div class="spin"></div><h2>Guwahati Urban Prediction</h2><div id="ls">Initializing...</div></div>
<div id="header"><div><h1>Guwahati Urban Development Prediction</h1><div class="sub">10-Year Growth Forecast (2026&ndash;2036) &middot; Landsat 30m</div></div><span class="badge">PoC v1.0</span></div>
<div id="map"></div>
<div id="sidebar">
<div class="ps"><div class="pt">LULC Classes</div>
<div class="lr"><div class="ls" style="background:var(--urban)"></div>Urban / Built-up</div>
<div class="lr"><div class="ls" style="background:var(--veg)"></div>Vegetation</div>
<div class="lr"><div class="ls" style="background:var(--water)"></div>Water</div>
<div class="lr"><div class="ls" style="background:var(--barren)"></div>Barren / Open</div>
</div>
<div class="ps"><div class="pt">Overlays</div><div id="lc"></div></div>
</div>
<div id="stats">
<div class="sc"><div class="sl">Urban Area</div><div class="sv" id="s-area">&mdash;</div><div class="sd" id="s-delta"></div></div>
<div class="sc"><div class="sl">Epoch</div><div class="sv" id="s-year">2021</div><div class="sd" id="s-type">Historical</div></div>
<div class="sc"><div class="pt">Urban Growth Trend</div><canvas id="chart"></canvas></div>
</div>
<div id="timeline">
<div id="tl-label">2021 <span class="st">Historical</span></div>
<input type="range" id="tl-slider" min="0" max="7" value="4" step="1">
<div id="tl-ticks"></div>
</div>
<script src="https://cdn.jsdelivr.net/npm/maplibre-gl@4.7.1/dist/maplibre-gl.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/pmtiles@3.2.0/dist/pmtiles.min.js"></script>
<script>
(function(){
"use strict";

var E = ['2000','2005','2010','2016','2021','2026','2031','2036'];
var C = [91.7362, 26.1445];
var B = [91.55, 26.1, 91.88, 26.25];
var OL = [
  {id:'urban_gain',l:'Urban Growth',d:false},
  {id:'suitability',l:'Growth Suitability',d:false},
  {id:'viirs',l:'Night Lights',d:false},
  {id:'ndvi',l:'Vegetation (NDVI)',d:false},
  {id:'elevation',l:'Elevation',d:false},
  {id:'slope',l:'Slope',d:false},
  {id:'osm_roads',l:'Roads (OSM)',d:false},
  {id:'osm_water',l:'Water (OSM)',d:false}
];
var M = null, map = null, ci = 4, hasLiveTiles = false;

function ss(m) {
  var e = document.getElementById('ls');
  if (e) e.textContent = m;
}

function showBanner(m) {
  var e = document.getElementById('banner');
  e.innerHTML = m;
  e.style.display = 'block';
}

function loadM() {
  ss('Loading tile data...');
  if (typeof INLINE_MANIFEST !== 'undefined' && INLINE_MANIFEST.tile_layers &&
      Object.keys(INLINE_MANIFEST.tile_layers).length > 0) {
    M = INLINE_MANIFEST;
    hasLiveTiles = true;
    console.log('Loaded inline manifest:', Object.keys(M.tile_layers).length, 'layers');
    return Promise.resolve();
  }
  return fetch('tile_manifest.json').then(function(r) {
    if (!r.ok) throw new Error('HTTP ' + r.status);
    return r.json();
  }).then(function(data) {
    if (data.tile_layers && Object.keys(data.tile_layers).length > 0) {
      M = data;
      hasLiveTiles = true;
    } else { throw new Error('Empty'); }
  }).catch(function(e) {
    console.warn('No live tiles:', e.message);
    M = {tile_layers:{}, urban_area_km2:{
      '2000':42.8,'2005':52.1,'2010':63.5,'2016':78.3,'2021':91.7,
      '2026':108.4,'2031':124.6,'2036':139.2
    }, vector_layers:{}};
    hasLiveTiles = false;
  });
}

function initMap() {
  ss('Loading map...');
  if (window.pmtiles) { var p = new pmtiles.Protocol(); maplibregl.addProtocol('pmtiles', p.tile); }
  map = new maplibregl.Map({
    container: 'map',
    style: { version: 8, sources: { 'esri': { type: 'raster',
      tiles: ['https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}'],
      tileSize: 256, attribution: 'Esri, Maxar', maxzoom: 18 } },
      layers: [{ id: 'base', type: 'raster', source: 'esri' }] },
    center: C, zoom: 12, maxBounds: [B[0]-.1, B[1]-.05, B[2]+.1, B[3]+.05]
  });
  map.addControl(new maplibregl.NavigationControl(), 'bottom-right');
  map.addControl(new maplibregl.ScaleControl({unit: 'metric'}), 'bottom-right');
  map.on('load', onLoad);
}

function onLoad() {
  ss('Adding layers...');
  var tl = M.tile_layers || {};

  E.forEach(function(y) {
    var k = 'lulc_' + y;
    if (tl[k] && tl[k].url) {
      map.addSource(k, {type: 'raster', tiles: [tl[k].url], tileSize: 256});
      map.addLayer({id: k, type: 'raster', source: k, paint: {'raster-opacity': 0}});
    }
  });

  Object.keys(tl).filter(function(k){return k.indexOf('urban_gain')===0}).forEach(function(k) {
    if (tl[k] && tl[k].url) {
      map.addSource(k, {type: 'raster', tiles: [tl[k].url], tileSize: 256});
      map.addLayer({id: k, type: 'raster', source: k, paint: {'raster-opacity': 0}});
    }
  });

  ['suitability_2021','ndvi_2021','elevation','slope'].forEach(function(k) {
    var sid = k.replace('_2021','');
    if (tl[k] && tl[k].url) {
      map.addSource(sid, {type: 'raster', tiles: [tl[k].url], tileSize: 256});
      map.addLayer({id: sid, type: 'raster', source: sid, paint: {'raster-opacity': 0}});
    }
  });

  [2023,2021,2019,2016].some(function(y) {
    var k = 'viirs_' + y;
    if (tl[k] && tl[k].url) {
      map.addSource('viirs', {type: 'raster', tiles: [tl[k].url], tileSize: 256});
      map.addLayer({id: 'viirs', type: 'raster', source: 'viirs', paint: {'raster-opacity': 0}});
      return true;
    }
  });

  var vl = M.vector_layers;
  if (vl && vl.osm && vl.osm.url) {
    var u = vl.osm.url.indexOf('http') === 0 ? vl.osm.url : 'pmtiles://' + vl.osm.url;
    try {
      map.addSource('osm', {type: 'vector', url: u});
      map.addLayer({id: 'osm_roads', type: 'line', source: 'osm', 'source-layer': 'roads',
        paint: {'line-color': '#ffd700', 'line-width': 1.5, 'line-opacity': 0}});
      map.addLayer({id: 'osm_water', type: 'line', source: 'osm', 'source-layer': 'water',
        paint: {'line-color': '#00bfff', 'line-width': 2, 'line-opacity': 0}});
    } catch(e) { console.warn('PMTiles:', e); }
  }

  buildCtrls(); buildTL(); updateEpoch(ci); drawChart();

  if (!hasLiveTiles) {
    showBanner('Demo mode &mdash; showing stats &amp; chart with sample data. ' +
      'For live LULC map tiles, run the Layer 5 Colab notebook and open the Colab preview URL.');
  }
  setTimeout(function(){ document.getElementById('loading').classList.add('h'); }, 600);
}

function buildCtrls() {
  var c = document.getElementById('lc');
  OL.forEach(function(o) {
    var l = document.createElement('label'); l.className = 'li';
    var cb = document.createElement('input'); cb.type = 'checkbox'; cb.checked = o.d;
    cb.onchange = function() {
      var p = o.id.indexOf('osm_') === 0 ? 'line-opacity' : 'raster-opacity';
      if (map.getLayer(o.id)) map.setPaintProperty(o.id, p, cb.checked ? 0.75 : 0);
    };
    l.appendChild(cb); l.appendChild(document.createTextNode(o.l)); c.appendChild(l);
  });
}

function buildTL() {
  var t = document.getElementById('tl-ticks');
  E.forEach(function(y, i) {
    var s = document.createElement('span'); s.textContent = y;
    if (+y > 2021) s.className = 'pred';
    s.onclick = function() { document.getElementById('tl-slider').value = i; updateEpoch(i); };
    t.appendChild(s);
  });
  document.getElementById('tl-slider').oninput = function(e) { updateEpoch(+e.target.value); };
}

function updateEpoch(i) {
  ci = i; var y = E[i], ip = +y > 2021;
  document.getElementById('tl-label').innerHTML =
    y + ' <span class="st">' + (ip ? 'Predicted' : 'Historical') + '</span>';
  E.forEach(function(yr, j) {
    var k = 'lulc_' + yr;
    if (map.getLayer(k)) map.setPaintProperty(k, 'raster-opacity', j === i ? 0.7 : 0);
  });
  document.getElementById('s-year').textContent = y;
  document.getElementById('s-type').textContent = ip ? 'CA-Markov + GBT' : 'Historical (CART)';
  var a = M.urban_area_km2 || {}, v = a[y];
  document.getElementById('s-area').textContent =
    v !== undefined ? v.toFixed(1) + ' km²' : '—';
  if (i > 0 && v !== undefined) {
    var pv = a[E[i-1]];
    if (pv !== undefined) {
      var d = v - pv, pct = ((d / pv) * 100).toFixed(1);
      var el = document.getElementById('s-delta');
      el.textContent = (d>0?'+':'') + d.toFixed(1) + ' km² (' + (d>0?'+':'') + pct + '%)';
      el.className = 'sd ' + (d > 0 ? 'up' : 'dn');
    }
  } else { document.getElementById('s-delta').textContent = ''; }
}

function drawChart() {
  var cv = document.getElementById('chart'), ctx = cv.getContext('2d');
  var a = M.urban_area_km2 || {}, dpr = devicePixelRatio || 1;
  cv.width = cv.offsetWidth * dpr; cv.height = cv.offsetHeight * dpr; ctx.scale(dpr, dpr);
  var w = cv.offsetWidth, h = cv.offsetHeight;
  var pad = {t:10, r:10, b:24, l:40}, cw = w - pad.l - pad.r, ch = h - pad.t - pad.b;
  var vals = E.map(function(y){ return a[y] != null ? a[y] : null; });
  var vv = vals.filter(function(v){ return v !== null; });
  if (vv.length < 2) {
    ctx.fillStyle = '#8a8a9a'; ctx.font = '11px sans-serif'; ctx.textAlign = 'center';
    ctx.fillText('No data available', w/2, h/2); return;
  }
  var mn = Math.min.apply(null, vv) * 0.9, mx = Math.max.apply(null, vv) * 1.1;
  ctx.strokeStyle = '#333'; ctx.lineWidth = 1;
  ctx.beginPath(); ctx.moveTo(pad.l, pad.t); ctx.lineTo(pad.l, h-pad.b); ctx.lineTo(w-pad.r, h-pad.b); ctx.stroke();
  ctx.fillStyle = '#8a8a9a'; ctx.font = '9px sans-serif'; ctx.textAlign = 'right';
  for (var i = 0; i <= 4; i++) {
    var v = mn + (mx - mn) * i / 4, yy = pad.t + ch - ch * i / 4;
    ctx.fillText(v.toFixed(0), pad.l - 4, yy + 3);
  }
  ctx.beginPath(); var started = false;
  E.forEach(function(yr, idx) {
    var val = vals[idx]; if (val === null) return;
    var x = pad.l + cw * idx / (E.length - 1);
    var y = pad.t + ch - ch * (val - mn) / (mx - mn);
    if (!started) { ctx.moveTo(x, y); started = true; } else ctx.lineTo(x, y);
  });
  ctx.strokeStyle = '#e94560'; ctx.lineWidth = 2; ctx.stroke();
  E.forEach(function(yr, idx) {
    var x = pad.l + cw * idx / (E.length - 1);
    ctx.fillStyle = +yr > 2021 ? '#533483' : '#8a8a9a';
    ctx.font = '8px sans-serif'; ctx.textAlign = 'center';
    ctx.fillText(yr.slice(2), x, h - pad.b + 12);
    var val = vals[idx]; if (val === null) return;
    var y = pad.t + ch - ch * (val - mn) / (mx - mn);
    ctx.beginPath(); ctx.arc(x, y, 3, 0, Math.PI * 2);
    ctx.fillStyle = +yr > 2021 ? '#e94560' : '#e0e0e0'; ctx.fill();
  });
  var dx = pad.l + cw * 4 / (E.length - 1);
  ctx.setLineDash([4, 4]); ctx.strokeStyle = '#533483'; ctx.lineWidth = 1;
  ctx.beginPath(); ctx.moveTo(dx, pad.t); ctx.lineTo(dx, h - pad.b); ctx.stroke(); ctx.setLineDash([]);
}

loadM().then(initMap);
})();
</script>
</body>
</html>'''

# Inject the manifest data directly into the HTML as an inline script
manifest_json = json.dumps(manifest, separators=(',', ':'))
manifest_tag = '<script>var INLINE_MANIFEST=' + manifest_json + ';</script>'
html_final = APP_HTML.replace('</head>', manifest_tag + '\n</head>')

with open('public/index.html', 'w') as f:
    f.write(html_final)

layer_count = len(manifest.get('tile_layers', {}))
print(f'Saved: public/index.html (with {layer_count} tile layers embedded inline)')
print(f'  Manifest size: {len(manifest_json)/1024:.0f} KB embedded')
print(f'  GEE tile URLs valid for ~24h from this Colab session')

# OSM PMTiles (if available from Layer 4)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    src = f'/content/drive/MyDrive/guwahati_urban_prediction'
    for fname in ['osm_guwahati.pmtiles', 'osm_roads.geojson', 'osm_water.geojson']:
        spath = f'{src}/{fname}'
        if os.path.exists(spath):
            shutil.copy(spath, f'public/data/{fname}')
            print(f'Copied from Drive: {fname}')
except:
    pass

# Firebase config
firebase_cfg = {"hosting":{"public":"public","ignore":["firebase.json","**/.*"],
    "headers":[{"source":"**/*.pmtiles","headers":[
        {"key":"Access-Control-Allow-Origin","value":"*"},
        {"key":"Cache-Control","value":"public, max-age=86400"}]},
    {"source":"**/*.json","headers":[
        {"key":"Access-Control-Allow-Origin","value":"*"},
        {"key":"Cache-Control","value":"public, max-age=3600"}]}]}}
with open('firebase.json', 'w') as f:
    json.dump(firebase_cfg, f, indent=2)
print('Saved: firebase.json')

print('\n=== Deployment Package ===')
for root, dirs, files in os.walk('public'):
    level = root.replace('public','').count(os.sep)
    print('  '*level + os.path.basename(root) + '/')
    for fn in files:
        sz = os.path.getsize(os.path.join(root, fn))/1024
        print('  '*(level+1) + f'{fn} ({sz:.0f} KB)')

print(f'''
=== How to view ===
  1. Run the next cell (Local Preview) to serve from Colab
  2. Open the Colab preview URL in a new tab
  3. The LULC layers will appear on the map — use the timeline slider
  
  Note: GEE tile URLs expire after ~24h.
  Re-run this notebook to refresh them.
''')

## 3. Local preview

In [ ]:
# Local preview — serve via Python HTTP server
# In Colab, use localtunnel or Colab's built-in port forwarding

import subprocess, threading, time

port = 8080

def serve():
    subprocess.run(['python', '-m', 'http.server', str(port), '--directory', 'public'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

server_thread = threading.Thread(target=serve, daemon=True)
server_thread.start()
time.sleep(1)

print(f'Local server running at http://localhost:{port}')
print()

# In Colab, use this to get a public URL:
try:
    from google.colab.output import eval_js
    url = eval_js(f'google.colab.kernel.proxyPort({port})')
    print(f'Colab preview URL: {url}')
    print('Open the URL above in a new tab to see the app.')
except:
    print('Not in Colab — open http://localhost:8080 in your browser.')
    print('Or run: python -m http.server 8080 --directory public')

## 4. Deploy to Firebase Hosting

In [ ]:
# Deploy to Firebase Hosting
# Prerequisites: npm install -g firebase-tools && firebase login

print('''
=== Firebase Hosting Deployment ===

Option A — Deploy from Colab:
  !npm install -g firebase-tools
  !firebase login --no-localhost
  !firebase init hosting  # select your project, set public dir to "public"
  !firebase deploy --only hosting

Option B — Deploy from local machine:
  1. Download the ZIP (next cell)
  2. Extract to your project directory
  3. Run:
     npm install -g firebase-tools
     firebase login
     firebase init hosting
     firebase deploy --only hosting

Option C — GitHub Pages:
  1. Push public/ contents to a gh-pages branch
  2. Enable Pages in repo settings

The app URL will be:
  Firebase: https://<project-id>.web.app
  GitHub:   https://<username>.github.io/<repo>/
''')

## 5. Download as ZIP (alternative)

In [ ]:
# Download deployment package as ZIP

import zipfile

zip_path = 'guwahati_urban_prediction_app.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('firebase.json', 'firebase.json')
    for root, dirs, files in os.walk('public'):
        for f in files:
            full = os.path.join(root, f)
            arcname = os.path.join(root, f)
            zf.write(full, arcname)

size_mb = os.path.getsize(zip_path) / 1e6
print(f'ZIP created: {zip_path} ({size_mb:.1f} MB)')

# Copy to Drive for persistence
try:
    dest = f'/content/drive/MyDrive/guwahati_urban_prediction/{zip_path}'
    shutil.copy(zip_path, dest)
    print(f'Copied to Drive: {dest}')
except:
    pass

# Download in Colab
try:
    from google.colab import files
    files.download(zip_path)
    print('Download started in browser.')
except:
    print(f'Download {zip_path} manually from the file browser.')